In [1]:
from pathlib import Path
import sys
import json
import csv
from collections import defaultdict

ROOT = Path.cwd().parents[1]
sys.path.insert(0, str(ROOT))

from sklearn.model_selection import train_test_split

In [2]:
from src.adapter.VHE_adapter import VHEAdapter
from src.unified_format.schema import Schema

In [3]:
geneva_adapter = VHEAdapter()

In [4]:
from pathlib import Path
import json

input_path = ROOT / "data/raw/VHE/event.json"
output_path = ROOT / "data/processed/VHE/event.jsonl"

adapter = VHEAdapter()

output_path.parent.mkdir(parents=True, exist_ok=True)

with open(input_path, "r", encoding="utf-8") as infile:
    data = json.load(infile)

with open(output_path, "w", encoding="utf-8") as outfile:
    for item in data:
        sample = adapter.adapt(item)

        result = json.loads(sample.to_json())

        outfile.write(
            json.dumps(result, ensure_ascii=False) + "\n"
        )

Event dvsktt-s5787 has 1 invalid spans. This event has been skipped.
Invalid spans for event dvsktt-s5787: ["Character span [7, 18] cuts a token: 'hạm Ngũ Lão'"]
Event dvsktt-s15095 has 1 invalid spans. This event has been skipped.
Invalid spans for event dvsktt-s15095: ["Character span [75, 115] cuts a token: 'ông các hiệu thư kiêm Tú lâm cục tư huấn'"]
Event dvsktt-s15769 has 1 invalid spans. This event has been skipped.
Invalid spans for event dvsktt-s15769: ["Character span [11, 20] cuts a token: 'guyễn Mậu'"]
Event dvsktt-s16909 has 1 invalid spans. This event has been skipped.
Invalid spans for event dvsktt-s16909: ["Character span [27, 40] cuts a token: 'cửa Yên Trườn'"]
Event dvsktt-s18887 has 1 invalid spans. This event has been skipped.
Invalid spans for event dvsktt-s18887: ["Character span [88, 107] cuts a token: 'án lý Phạm Công Trứ'"]
Event dvsktt-s19857 has 1 invalid spans. This event has been skipped.
Invalid spans for event dvsktt-s19857: ["Character span [47, 52] cuts

### 8-1-1

In [5]:
input_path = ROOT / "data/processed/VHE/event.jsonl"
output_dir = ROOT / "data/processed/VHE"

with open(input_path, "r", encoding="utf-8") as file:
    data = [json.loads(line) for line in file if line.strip()]

# 80% train, 20% temporary
train_data, temp_data = train_test_split(
    data,
    test_size=0.2,
    random_state=42
)

# Chia 20% còn lại thành 10% dev, 10% test
dev_data, test_data = train_test_split(
    temp_data,
    test_size=0.5,
    random_state=42
)

splits = {
    "train": train_data,
    "dev": dev_data,
    "test": test_data,
}

for split, samples in splits.items():
    output_path = output_dir / f"{split}.jsonl"

    with open(output_path, "w", encoding="utf-8") as file:
        for sample in samples:
            file.write(
                json.dumps(sample, ensure_ascii=False) + "\n"
            )

    print(f"{split}: {len(samples)} samples")

train: 3288 samples
dev: 411 samples
test: 412 samples


In [6]:
import json
from dataclasses import asdict

input_path = ROOT / "data/raw/VHE/vhe_event_schema.json"
output_path = ROOT / "data/processed/VHE/schema.json"

with open(input_path, "r", encoding="utf-8") as file:
    data = [json.loads(line) for line in file]

event_schemas = [
    adapter.get_schema(item)
    for item in data
]

schema = Schema(event_schemas=event_schemas)

output_path.parent.mkdir(parents=True, exist_ok=True)

with open(output_path, "w", encoding="utf-8") as file:
    json.dump(
        asdict(schema),
        file,
        ensure_ascii=False,
        indent=2
    )